# 04 - Florence-2 + SAM: prompts, tamaño de modelo y tiling

Detección con Florence-2 y segmentación con SAM (prompt de caja) sobre las 137
imágenes de WGISD que tienen máscara.

- **Preparación**: se corre una vez por sesión.
- **Pruebas**: cada celda es una prueba y se puede correr sola, en cualquier
  orden. Guarda sus métricas en `outputs/04_florence_sam_prompts/pruebas/`, un
  archivo por variante.
- **Comparación**: lee todo lo guardado, también lo de sesiones anteriores.

Las métricas de detección usan matching húngaro sobre IoU de caja, con umbral 0.5.

## Preparación

In [1]:
import json
import re
import sys
import time
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from segment_anything import SamPredictor, sam_model_registry
from transformers import AutoProcessor, Florence2ForConditionalGeneration

from grape_autoseg.data.wgisd import list_masked_names, load_sample
from grape_autoseg.detection.tiling import box_areas, detect_tiled, drop_group_boxes, merge_boxes, merge_masks
from grape_autoseg.evaluation.detection_metrics import aggregate_detection_results, evaluate_detection
from grape_autoseg.evaluation.mask_metrics import aggregate_mask_results, evaluate_masks_for_matches

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WGISD_IMAGES_DIR = Path("../data/wgisd/data")
SAM_CHECKPOINTS = {
    "vit_b": "../data/checkpoints/sam_vit_b_01ec64.pth",
    "vit_h": "../data/checkpoints/sam_vit_h_4b8939.pth",
}
OUT_DIR = Path("../outputs/04_florence_sam_prompts")
RESULTS_DIR = OUT_DIR / "pruebas"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IOU_THRESHOLD = 0.5
TILE_COLS, TILE_ROWS, TILE_OVERLAP = 4, 3, 0.2
MASK_STRIDE = 8  # las máscaras se guardan submuestreadas: a resolución completa no entran todas en memoria

print("device:", DEVICE)

device: cuda


### Modelos

Florence-2 y SAM se cargan la primera vez que una prueba pide ese tamaño.

In [3]:
_florence, _sam = {}, {}


def get_florence(size):
    if size not in _florence:
        model_id = f"florence-community/Florence-2-{size}"
        model = Florence2ForConditionalGeneration.from_pretrained(model_id, device_map=DEVICE).eval()
        _florence[size] = (model, AutoProcessor.from_pretrained(model_id))
    return _florence[size]


def get_sam(sam_model):
    if sam_model not in _sam:
        _sam[sam_model] = SamPredictor(sam_model_registry[sam_model](checkpoint=SAM_CHECKPOINTS[sam_model]).to(DEVICE))
    return _sam[sam_model]

### Ground truth

Cajas y máscaras de las imágenes con máscara. Las máscaras quedan submuestreadas (`MASK_STRIDE`).

In [4]:
masked_names = list_masked_names(WGISD_IMAGES_DIR)

gt_boxes, gt_masks = {}, {}
for name in masked_names:
    sample = load_sample(WGISD_IMAGES_DIR, name)
    gt_boxes[name] = sample.boxes_xyxy
    gt_masks[name] = np.moveaxis(sample.masks[::MASK_STRIDE, ::MASK_STRIDE] > 0, -1, 0)
del sample

print(f"{len(masked_names)} imagenes con mascara, {sum(len(b) for b in gt_boxes.values())} racimos")

137 imagenes con mascara, 2019 racimos


### Detección y segmentación

`detect_full`, `detect_tiles` y `segment` guardan su resultado en memoria
(`_memo`), así dos pruebas que usan las mismas cajas o máscaras no repiten la
pasada. Si se cambia `florence_detect`, vaciar con `_memo.clear()`.

In [5]:
_memo = {}


def memo(key, compute):
    if key not in _memo:
        _memo[key] = compute()
    return _memo[key]


def each_image(label):
    t_start = time.time()
    for i, name in enumerate(masked_names):
        yield name
        if (i + 1) % 20 == 0:
            print(f"{label}: [{i+1}/{len(masked_names)}] {(time.time()-t_start)/60:.1f}min")
    print(f"{label}: listo en {(time.time()-t_start)/60:.1f}min")


def load_image(name):
    return Image.open(WGISD_IMAGES_DIR / f"{name}.jpg").convert("RGB")


def florence_detect(image, phrase, size="base", num_beams=1):
    model, processor = get_florence(size)
    task_prompt = "<OPEN_VOCABULARY_DETECTION>"
    inputs = processor(text=task_prompt + phrase, images=image, return_tensors="pt").to(DEVICE, torch.float32)
    with torch.no_grad():
        # cada caja son 4 tokens: con menos de ~150 la lista de cajas se corta
        ids = model.generate(**inputs, max_new_tokens=256, num_beams=num_beams)
    text = processor.batch_decode(ids, skip_special_tokens=False)[0]
    parsed = processor.post_process_generation(text, task=task_prompt, image_size=image.size)
    boxes = parsed[task_prompt]["bboxes"]
    return np.array(boxes).reshape(-1, 4) if boxes else np.zeros((0, 4))


def detect_full(phrase="grapes", size="base", num_beams=1):
    """Cajas de Florence sobre la imagen completa: {imagen: cajas}."""
    label = f"Florence-{size} '{phrase}' beams={num_beams}, imagen completa"
    return memo(
        ("full", phrase, size, num_beams),
        lambda: {name: florence_detect(load_image(name), phrase, size, num_beams) for name in each_image(label)},
    )


def detect_tiles(phrase="grapes", size="base", num_beams=1):
    """Cajas crudas (sin fusionar) de los tiles, en coordenadas de la imagen completa: {imagen: cajas}."""
    label = f"Florence-{size} '{phrase}' beams={num_beams}, tiles {TILE_COLS}x{TILE_ROWS}"

    def compute():
        return {
            name: detect_tiled(
                load_image(name), lambda tile: florence_detect(tile, phrase, size, num_beams), TILE_COLS, TILE_ROWS, TILE_OVERLAP
            )[0]
            for name in each_image(label)
        }

    return memo(("tiles", phrase, size, num_beams), compute)


def fuse_tiles(tiles, full=None, min_side=0):
    """Fusiona las cajas de tiles con NMS de caja. `min_side` descarta antes las de lado equivalente menor."""
    fused = {}
    for name, boxes in tiles.items():
        if len(boxes):
            boxes = boxes[np.sqrt(box_areas(boxes)) >= min_side]
        if full is not None:
            boxes = np.concatenate([boxes, full[name]])
        fused[name] = boxes[merge_boxes(boxes, iou_threshold=0.5)]
    return fused

In [6]:
def sam_masks_for_boxes(image_np, boxes_xyxy, sam_model="vit_b"):
    """SAM con prompt de caja. Devuelve, por cada caja con máscara no vacía:
    caja ajustada a la máscara, score de SAM, máscara submuestreada e índice de la caja de entrada."""
    sam_predictor = get_sam(sam_model)
    src_idx, fit_boxes, scores, small_masks = [], [], [], []
    if len(boxes_xyxy) > 0:
        sam_predictor.set_image(image_np)
    for k, box in enumerate(boxes_xyxy):
        m, s, _ = sam_predictor.predict(box=np.array(box), multimask_output=False)
        small = m[0][::MASK_STRIDE, ::MASK_STRIDE]
        if not small.any():
            continue
        ys, xs = np.nonzero(m[0])
        src_idx.append(k)
        fit_boxes.append([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1])
        scores.append(float(s[0]))
        small_masks.append(small)

    if not src_idx:
        h, w = image_np[::MASK_STRIDE, ::MASK_STRIDE].shape[:2]
        return {"boxes": np.zeros((0, 4)), "scores": np.zeros(0), "masks": np.zeros((0, h, w), dtype=bool), "src_idx": np.zeros(0, dtype=int)}
    return {"boxes": np.array(fit_boxes, dtype=float), "scores": np.array(scores), "masks": np.stack(small_masks), "src_idx": np.array(src_idx)}


def segment(source, phrase="grapes", size="base", num_beams=1, sam_model="vit_b"):
    """Máscaras de SAM para las cajas de `source` ("full" o "tiles"): {imagen: salida de sam_masks_for_boxes}."""
    detect = {"full": detect_full, "tiles": detect_tiles}[source]

    def compute():
        boxes = detect(phrase, size, num_beams)
        return {
            name: sam_masks_for_boxes(np.array(load_image(name)), boxes[name], sam_model)
            for name in each_image(f"SAM {sam_model} sobre cajas de {source}")
        }

    return memo(("sam", source, phrase, size, num_beams, sam_model), compute)


def dedup_masks(segments, containment=None, group_min_fraction=None):
    """NMS de máscara sobre una o más salidas de `segment`. Devuelve ({imagen: cajas}, {imagen: máscaras})."""
    boxes, masks = {}, {}
    for name in masked_names:
        seg = {k: np.concatenate([s[name][k] for s in segments]) for k in ("boxes", "scores", "masks")}
        keep = merge_masks(
            seg["masks"], seg["scores"], iou_threshold=0.5, containment_threshold=containment, group_min_fraction=group_min_fraction
        )
        boxes[name], masks[name] = seg["boxes"][keep], seg["masks"][keep]
    return boxes, masks


DEDUP_VARIANTS = {
    "NMS de mascara": {},
    "NMS de mascara + contencion": {"containment": 0.7},
    "NMS de mascara + contencion sin grupos": {"containment": 0.7, "group_min_fraction": 0.2},
}

### Evaluación y guardado

Métricas de detección:

- `micro_*`: sobre el total de cajas. `macro_f1`: promedio simple por imagen.
- `mae_count`: error absoluto medio de conteo por imagen.

Métricas de máscara (solo en las variantes con SAM), a la resolución submuestreada:

- `px_precision`, `px_recall`, `dice`, `union_mask_iou`: entre la unión de todas
  las máscaras predichas y la unión de todas las máscaras GT, sumando píxeles
  sobre todas las imágenes. `dice` es el F1 sobre píxeles.
- `mean_mask_iou_matched`: IoU de máscara promedio de los pares que hicieron
  match a nivel caja. `pct_mask_iou_above_75`: fracción de esos pares con IoU ≥ 0.75.

In [7]:
def evaluate(pred_boxes, pred_masks=None):
    det_results, mask_results = [], []
    inter_px = pred_px = gt_px = 0
    for name in masked_names:
        det = evaluate_detection(name, pred_boxes[name], gt_boxes[name], iou_threshold=IOU_THRESHOLD)
        det_results.append(det)
        if pred_masks is not None:
            mask_results.append(evaluate_masks_for_matches(name, pred_masks[name], gt_masks[name], det.matches))
            pred_union, gt_union = pred_masks[name].any(axis=0), gt_masks[name].any(axis=0)
            inter_px += int((pred_union & gt_union).sum())
            pred_px += int(pred_union.sum())
            gt_px += int(gt_union.sum())

    agg = aggregate_detection_results(det_results)
    row = {
        "micro_precision": agg["micro_precision"],
        "micro_recall": agg["micro_recall"],
        "micro_f1": agg["micro_f1"],
        "macro_f1": agg["macro_f1"],
        "mae_count": agg["mae_count"],
        "mean_box_iou_matched": agg["mean_match_iou"],
        "total_pred": agg["total_pred"],
        "total_gt": agg["total_gt"],
    }
    if pred_masks is not None:
        mask_agg = aggregate_mask_results(mask_results)
        row.update(
            {
                "px_precision": inter_px / pred_px if pred_px else 0.0,
                "px_recall": inter_px / gt_px if gt_px else 0.0,
                "dice": 2 * inter_px / (pred_px + gt_px) if pred_px + gt_px else 0.0,
                "union_mask_iou": inter_px / (pred_px + gt_px - inter_px) if pred_px + gt_px else 0.0,
                "mean_mask_iou_matched": mask_agg["mean_iou"],
                "pct_mask_iou_above_75": mask_agg["pct_above_75"],
            }
        )
    return row


def save_result(prueba, variante, pred_boxes, pred_masks=None, phrase="grapes", size="base", num_beams=1, sam_model="vit_b"):
    """Evalúa una variante y guarda sus métricas en `RESULTS_DIR`. Repetirla reemplaza su archivo."""
    row = {
        "prueba": prueba,
        "variante": variante,
        "florence_size": size,
        "prompt": phrase,
        "num_beams": num_beams,
        "sam_model": sam_model if pred_masks is not None else None,
        **evaluate(pred_boxes, pred_masks),
    }
    filename = re.sub(r"[^\w=+.-]+", "_", f"{prueba}__{variante}__{size}__{phrase}__beams{num_beams}")
    if pred_masks is not None and sam_model != "vit_b":
        filename += f"__sam_{sam_model}"
    (RESULTS_DIR / f"{filename}.json").write_text(json.dumps(row, indent=2))
    return row


TABLE_COLUMNS = [
    "prueba", "variante", "florence_size", "prompt", "num_beams", "sam_model",
    "micro_precision", "micro_recall", "micro_f1", "mae_count", "total_pred",
    "px_precision", "px_recall", "dice", "union_mask_iou", "mean_mask_iou_matched",
]


def show(rows):
    return pd.DataFrame(rows).reindex(columns=TABLE_COLUMNS).dropna(axis=1, how="all").round(3)

## Pruebas

### Prompts con Florence-2 `base`

In [8]:
cfg = {"size": "base", "num_beams": 1}
PROMPTS = ["grapes", "grape cluster", "wine grapes", "grape"]

show([save_result("prompts", "imagen completa", detect_full(phrase=p, **cfg), phrase=p, **cfg) for p in PROMPTS])

Loading weights:   0%|          | 0/665 [00:00<?, ?it/s]

Florence-base 'grapes' beams=1, imagen completa: [20/137] 0.2min
Florence-base 'grapes' beams=1, imagen completa: [40/137] 0.3min
Florence-base 'grapes' beams=1, imagen completa: [60/137] 0.3min
Florence-base 'grapes' beams=1, imagen completa: [80/137] 0.4min
Florence-base 'grapes' beams=1, imagen completa: [100/137] 0.5min
Florence-base 'grapes' beams=1, imagen completa: [120/137] 0.6min
Florence-base 'grapes' beams=1, imagen completa: listo en 0.7min
Florence-base 'grape cluster' beams=1, imagen completa: [20/137] 0.0min
Florence-base 'grape cluster' beams=1, imagen completa: [40/137] 0.1min
Florence-base 'grape cluster' beams=1, imagen completa: [60/137] 0.1min
Florence-base 'grape cluster' beams=1, imagen completa: [80/137] 0.2min
Florence-base 'grape cluster' beams=1, imagen completa: [100/137] 0.2min
Florence-base 'grape cluster' beams=1, imagen completa: [120/137] 0.3min
Florence-base 'grape cluster' beams=1, imagen completa: listo en 0.3min
Florence-base 'wine grapes' beams=1, 

,prueba,variante,florence_size,prompt,num_beams,micro_precision,micro_recall,micro_f1,mae_count,total_pred
0,prompts,imagen completa,base,grapes,1,0.580,0.418,0.486,4.599,1453
1,prompts,imagen completa,base,grape cluster,1,0.435,0.059,0.105,12.723,276
2,prompts,imagen completa,base,wine grapes,1,0.418,0.136,0.205,9.956,655
3,prompts,imagen completa,base,grape,1,0.587,0.342,0.432,6.321,1177


### Prompts con Florence-2 `large`

In [9]:
cfg = {"size": "large", "num_beams": 1}
PROMPTS = ["grapes", "grape cluster", "wine grapes", "grape"]

show([save_result("prompts", "imagen completa", detect_full(phrase=p, **cfg), phrase=p, **cfg) for p in PROMPTS])

Loading weights:   0%|          | 0/917 [00:00<?, ?it/s]

Florence-large 'grapes' beams=1, imagen completa: [20/137] 0.2min
Florence-large 'grapes' beams=1, imagen completa: [40/137] 0.3min
Florence-large 'grapes' beams=1, imagen completa: [60/137] 0.5min
Florence-large 'grapes' beams=1, imagen completa: [80/137] 0.6min
Florence-large 'grapes' beams=1, imagen completa: [100/137] 0.8min
Florence-large 'grapes' beams=1, imagen completa: [120/137] 0.9min
Florence-large 'grapes' beams=1, imagen completa: listo en 1.0min
Florence-large 'grape cluster' beams=1, imagen completa: [20/137] 0.1min
Florence-large 'grape cluster' beams=1, imagen completa: [40/137] 0.1min
Florence-large 'grape cluster' beams=1, imagen completa: [60/137] 0.2min
Florence-large 'grape cluster' beams=1, imagen completa: [80/137] 0.3min
Florence-large 'grape cluster' beams=1, imagen completa: [100/137] 0.3min
Florence-large 'grape cluster' beams=1, imagen completa: [120/137] 0.4min
Florence-large 'grape cluster' beams=1, imagen completa: listo en 0.4min
Florence-large 'wine gr

,prueba,variante,florence_size,prompt,num_beams,micro_precision,micro_recall,micro_f1,mae_count,total_pred
0,prompts,imagen completa,large,grapes,1,0.556,0.345,0.426,5.920,1252
1,prompts,imagen completa,large,grape cluster,1,0.482,0.033,0.061,13.737,137
2,prompts,imagen completa,large,wine grapes,1,0.288,0.035,0.062,12.993,243
3,prompts,imagen completa,large,grape,1,0.541,0.138,0.220,11.029,516


### Beam search (`num_beams=3`)

In [10]:
cfg = {"phrase": "grapes", "size": "base", "num_beams": 3}

show([save_result("prompts", "imagen completa", detect_full(**cfg), **cfg)])

Florence-base 'grapes' beams=3, imagen completa: [20/137] 0.1min
Florence-base 'grapes' beams=3, imagen completa: [40/137] 0.2min
Florence-base 'grapes' beams=3, imagen completa: [60/137] 0.3min
Florence-base 'grapes' beams=3, imagen completa: [80/137] 0.5min
Florence-base 'grapes' beams=3, imagen completa: [100/137] 0.6min
Florence-base 'grapes' beams=3, imagen completa: [120/137] 0.7min
Florence-base 'grapes' beams=3, imagen completa: listo en 0.8min


,prueba,variante,florence_size,prompt,num_beams,micro_precision,micro_recall,micro_f1,mae_count,total_pred
0,prompts,imagen completa,base,grapes,3,0.587,0.421,0.49,4.584,1447


### Tiling 4x3, fusión por cajas

Grilla de 4 columnas x 3 filas con 20 % de solape. Variantes:

- `min_side`: descarta cajas de tile con lado equivalente menor a ese valor.
- `+ imagen completa`: suma las cajas de la pasada global antes de fusionar.
- La fusión es NMS por IoU 0.5 con prioridad por área (`merge_boxes`).

In [11]:
cfg = {"phrase": "grapes", "size": "base", "num_beams": 1}
full, tiles = detect_full(**cfg), detect_tiles(**cfg)

rows = []
for min_side in [0, 60, 100]:
    rows.append(save_result("tiling cajas", f"tiles 4x3, min_side={min_side}", fuse_tiles(tiles, min_side=min_side), **cfg))
    rows.append(
        save_result("tiling cajas", f"tiles 4x3 + imagen completa, min_side={min_side}", fuse_tiles(tiles, full, min_side), **cfg)
    )
show(rows)

Florence-base 'grapes' beams=1, tiles 4x3: [20/137] 0.6min
Florence-base 'grapes' beams=1, tiles 4x3: [40/137] 1.5min
Florence-base 'grapes' beams=1, tiles 4x3: [60/137] 2.2min
Florence-base 'grapes' beams=1, tiles 4x3: [80/137] 3.1min
Florence-base 'grapes' beams=1, tiles 4x3: [100/137] 3.6min
Florence-base 'grapes' beams=1, tiles 4x3: [120/137] 4.3min
Florence-base 'grapes' beams=1, tiles 4x3: listo en 5.9min


,prueba,variante,florence_size,prompt,num_beams,micro_precision,micro_recall,micro_f1,mae_count,total_pred
0,tiling cajas,"tiles 4x3, min_side=0",base,grapes,1,0.118,0.545,0.195,53.139,9299
1,tiling cajas,"tiles 4x3 + imagen completa, min_side=0",base,grapes,1,0.125,0.598,0.207,55.635,9641
2,tiling cajas,"tiles 4x3, min_side=60",base,grapes,1,0.338,0.537,0.415,8.737,3208
3,tiling cajas,"tiles 4x3 + imagen completa, min_side=60",base,grapes,1,0.335,0.592,0.428,11.292,3566
4,tiling cajas,"tiles 4x3, min_side=100",base,grapes,1,0.383,0.485,0.428,4.723,2558
5,tiling cajas,"tiles 4x3 + imagen completa, min_side=100",base,grapes,1,0.380,0.566,0.455,7.409,3006


### Imagen completa + SAM

SAM con prompt de caja para cada caja de Florence, y deduplicación sobre las máscaras:

- **NMS de máscara**: por IoU 0.5, con prioridad por el score de SAM (`merge_masks`).
- **contención**: además descarta una máscara si el 70 % de su área está dentro
  de otra más grande.
- **contención sin grupos**: antes de eso, si una máscara contiene dos o más
  máscaras que ocupan cada una al menos el 20 % de su área, se la considera un
  grupo de racimos y se descarta la de afuera en vez de las de adentro.

Las métricas de detección se calculan sobre la caja ajustada a cada máscara que queda.

In [12]:
cfg = {"phrase": "grapes", "size": "base", "num_beams": 1}
segments = [segment("full", **cfg)]

show([save_result("imagen completa + SAM", label, *dedup_masks(segments, **kw), **cfg) for label, kw in DEDUP_VARIANTS.items()])

SAM vit_b sobre cajas de full: [20/137] 0.1min
SAM vit_b sobre cajas de full: [40/137] 0.3min
SAM vit_b sobre cajas de full: [60/137] 0.7min
SAM vit_b sobre cajas de full: [100/137] 1.5min
SAM vit_b sobre cajas de full: [120/137] 1.6min
SAM vit_b sobre cajas de full: listo en 1.7min


,prueba,variante,florence_size,prompt,num_beams,sam_model,micro_precision,micro_recall,micro_f1,mae_count,total_pred,px_precision,px_recall,dice,union_mask_iou,mean_mask_iou_matched
0,imagen completa + SAM,NMS de mascara,base,grapes,1,vit_b,0.626,0.410,0.496,5.299,1323,0.715,0.923,0.806,0.675,0.817
1,imagen completa + SAM,NMS de mascara + contencion,base,grapes,1,vit_b,0.587,0.283,0.382,7.708,975,0.716,0.914,0.803,0.671,0.811
2,imagen completa + SAM,NMS de mascara + contencion sin grupos,base,grapes,1,vit_b,0.629,0.331,0.434,7.058,1064,0.723,0.907,0.805,0.673,0.812


In [13]:
cfg = {"phrase": "grapes", "size": "base", "num_beams": 3}
segments = [segment("full", **cfg)]

show([save_result("imagen completa + SAM", label, *dedup_masks(segments, **kw), **cfg) for label, kw in DEDUP_VARIANTS.items()])

SAM vit_b sobre cajas de full: [20/137] 0.1min
SAM vit_b sobre cajas de full: [40/137] 0.2min
SAM vit_b sobre cajas de full: [60/137] 0.3min
SAM vit_b sobre cajas de full: [80/137] 0.4min
SAM vit_b sobre cajas de full: [100/137] 0.5min
SAM vit_b sobre cajas de full: [120/137] 0.6min
SAM vit_b sobre cajas de full: listo en 0.7min


,prueba,variante,florence_size,prompt,num_beams,sam_model,micro_precision,micro_recall,micro_f1,mae_count,total_pred,px_precision,px_recall,dice,union_mask_iou,mean_mask_iou_matched
0,imagen completa + SAM,NMS de mascara,base,grapes,3,vit_b,0.628,0.412,0.498,5.255,1325,0.734,0.922,0.817,0.691,0.817
1,imagen completa + SAM,NMS de mascara + contencion,base,grapes,3,vit_b,0.589,0.288,0.387,7.599,988,0.735,0.914,0.815,0.688,0.811
2,imagen completa + SAM,NMS de mascara + contencion sin grupos,base,grapes,3,vit_b,0.633,0.339,0.441,6.927,1080,0.743,0.906,0.817,0.690,0.813


### Tiles + SAM

In [14]:
cfg = {"phrase": "grapes", "size": "base", "num_beams": 1}
segments = [segment("tiles", **cfg)]

show([save_result("tiles + SAM", label, *dedup_masks(segments, **kw), **cfg) for label, kw in DEDUP_VARIANTS.items()])

SAM vit_b sobre cajas de tiles: [20/137] 0.3min
SAM vit_b sobre cajas de tiles: [40/137] 0.7min
SAM vit_b sobre cajas de tiles: [60/137] 0.9min
SAM vit_b sobre cajas de tiles: [80/137] 1.3min
SAM vit_b sobre cajas de tiles: [100/137] 1.5min
SAM vit_b sobre cajas de tiles: [120/137] 1.8min
SAM vit_b sobre cajas de tiles: listo en 2.2min


,prueba,variante,florence_size,prompt,num_beams,sam_model,micro_precision,micro_recall,micro_f1,mae_count,total_pred,px_precision,px_recall,dice,union_mask_iou,mean_mask_iou_matched
0,tiles + SAM,NMS de mascara,base,grapes,1,vit_b,0.126,0.565,0.206,51.248,9040,0.597,0.876,0.710,0.550,0.774
1,tiles + SAM,NMS de mascara + contencion,base,grapes,1,vit_b,0.247,0.426,0.313,12.109,3486,0.596,0.864,0.706,0.545,0.775
2,tiles + SAM,NMS de mascara + contencion sin grupos,base,grapes,1,vit_b,0.259,0.480,0.337,13.124,3735,0.597,0.856,0.704,0.543,0.781


In [15]:
cfg = {"phrase": "grapes", "size": "base", "num_beams": 3}
segments = [segment("tiles", **cfg)]

show([save_result("tiles + SAM", label, *dedup_masks(segments, **kw), **cfg) for label, kw in DEDUP_VARIANTS.items()])

Florence-base 'grapes' beams=3, tiles 4x3: [20/137] 0.9min
Florence-base 'grapes' beams=3, tiles 4x3: [40/137] 2.3min
Florence-base 'grapes' beams=3, tiles 4x3: [60/137] 3.2min
Florence-base 'grapes' beams=3, tiles 4x3: [80/137] 4.6min
Florence-base 'grapes' beams=3, tiles 4x3: [100/137] 5.3min
Florence-base 'grapes' beams=3, tiles 4x3: [120/137] 6.2min
Florence-base 'grapes' beams=3, tiles 4x3: listo en 7.3min
SAM vit_b sobre cajas de tiles: [20/137] 0.3min
SAM vit_b sobre cajas de tiles: [40/137] 0.7min
SAM vit_b sobre cajas de tiles: [60/137] 1.0min
SAM vit_b sobre cajas de tiles: [80/137] 1.4min
SAM vit_b sobre cajas de tiles: [100/137] 1.6min
SAM vit_b sobre cajas de tiles: [120/137] 1.9min
SAM vit_b sobre cajas de tiles: listo en 2.3min


,prueba,variante,florence_size,prompt,num_beams,sam_model,micro_precision,micro_recall,micro_f1,mae_count,total_pred,px_precision,px_recall,dice,union_mask_iou,mean_mask_iou_matched
0,tiles + SAM,NMS de mascara,base,grapes,3,vit_b,0.122,0.565,0.201,53.226,9309,0.603,0.878,0.715,0.557,0.773
1,tiles + SAM,NMS de mascara + contencion,base,grapes,3,vit_b,0.236,0.432,0.305,13.745,3706,0.602,0.866,0.711,0.551,0.773
2,tiles + SAM,NMS de mascara + contencion sin grupos,base,grapes,3,vit_b,0.248,0.486,0.329,14.876,3959,0.603,0.858,0.708,0.548,0.778


### Tiles + imagen completa + SAM

In [16]:
cfg = {"phrase": "grapes", "size": "base", "num_beams": 1}
segments = [segment("tiles", **cfg), segment("full", **cfg)]

show([save_result("tiles + imagen completa + SAM", label, *dedup_masks(segments, **kw), **cfg) for label, kw in DEDUP_VARIANTS.items()])

,prueba,variante,florence_size,prompt,num_beams,sam_model,micro_precision,micro_recall,micro_f1,mae_count,total_pred,px_precision,px_recall,dice,union_mask_iou,mean_mask_iou_matched
0,tiles + imagen completa + SAM,NMS de mascara,base,grapes,1,vit_b,0.133,0.611,0.218,53.088,9292,0.540,0.964,0.693,0.530,0.787
1,tiles + imagen completa + SAM,NMS de mascara + contencion,base,grapes,1,vit_b,0.266,0.319,0.290,6.745,2427,0.540,0.945,0.688,0.524,0.798
2,tiles + imagen completa + SAM,NMS de mascara + contencion sin grupos,base,grapes,1,vit_b,0.303,0.454,0.364,9.022,3019,0.543,0.919,0.682,0.518,0.792


In [17]:
cfg = {"phrase": "grapes", "size": "base", "num_beams": 3}
segments = [segment("tiles", **cfg), segment("full", **cfg)]

show([save_result("tiles + imagen completa + SAM", label, *dedup_masks(segments, **kw), **cfg) for label, kw in DEDUP_VARIANTS.items()])

,prueba,variante,florence_size,prompt,num_beams,sam_model,micro_precision,micro_recall,micro_f1,mae_count,total_pred,px_precision,px_recall,dice,union_mask_iou,mean_mask_iou_matched
0,tiles + imagen completa + SAM,NMS de mascara,base,grapes,3,vit_b,0.129,0.611,0.213,55.029,9558,0.552,0.962,0.701,0.540,0.788
1,tiles + imagen completa + SAM,NMS de mascara + contencion,base,grapes,3,vit_b,0.261,0.326,0.290,7.336,2522,0.552,0.944,0.696,0.534,0.797
2,tiles + imagen completa + SAM,NMS de mascara + contencion sin grupos,base,grapes,3,vit_b,0.303,0.459,0.365,9.526,3056,0.554,0.918,0.691,0.528,0.794


### Imagen completa sin cajas de grupo

Antes de SAM se sacan las cajas de grupo de la imagen completa: una caja que
contiene dos o más cajas predichas (`drop_group_boxes`). Primero solo con las
cajas de Florence, y después pasando las que quedan por SAM + NMS de máscara.

In [18]:
cfg = {"phrase": "grapes", "size": "base", "num_beams": 1}
full = detect_full(**cfg)
keep = {name: drop_group_boxes(boxes) for name, boxes in full.items()}

rows = [save_result("sin cajas de grupo", "cajas de Florence", {name: full[name][keep[name]] for name in full}, **cfg)]

segments_kept = {
    name: {k: v[np.isin(seg["src_idx"], keep[name])] for k, v in seg.items()} for name, seg in segment("full", **cfg).items()
}
rows.append(save_result("sin cajas de grupo", "SAM, NMS de mascara", *dedup_masks([segments_kept]), **cfg))
show(rows)

,prueba,variante,florence_size,prompt,num_beams,sam_model,micro_precision,micro_recall,micro_f1,mae_count,total_pred,px_precision,px_recall,dice,union_mask_iou,mean_mask_iou_matched
0,sin cajas de grupo,cajas de Florence,base,grapes,1,None,0.676,0.417,0.515,5.803,1244,NaN,NaN,NaN,NaN,NaN
1,sin cajas de grupo,"SAM, NMS de mascara",base,grapes,1,vit_b,0.696,0.414,0.519,6.117,1201,0.866,0.837,0.851,0.741,0.816


In [19]:
cfg = {"phrase": "grapes", "size": "base", "num_beams": 3}
full = detect_full(**cfg)
keep = {name: drop_group_boxes(boxes) for name, boxes in full.items()}

rows = [save_result("sin cajas de grupo", "cajas de Florence", {name: full[name][keep[name]] for name in full}, **cfg)]

segments_kept = {
    name: {k: v[np.isin(seg["src_idx"], keep[name])] for k, v in seg.items()} for name, seg in segment("full", **cfg).items()
}
rows.append(save_result("sin cajas de grupo", "SAM, NMS de mascara", *dedup_masks([segments_kept]), **cfg))
show(rows)

,prueba,variante,florence_size,prompt,num_beams,sam_model,micro_precision,micro_recall,micro_f1,mae_count,total_pred,px_precision,px_recall,dice,union_mask_iou,mean_mask_iou_matched
0,sin cajas de grupo,cajas de Florence,base,grapes,3,None,0.676,0.420,0.518,5.723,1255,NaN,NaN,NaN,NaN,NaN
1,sin cajas de grupo,"SAM, NMS de mascara",base,grapes,3,vit_b,0.690,0.415,0.518,6.022,1214,0.867,0.845,0.856,0.748,0.818


### SAM `vit_h`: imagen completa + SAM con NMS de máscara

In [20]:
SAM_MODEL = "vit_h"
cfg = {"phrase": "grapes", "size": "base", "num_beams": 3}

segments = [segment("full", sam_model=SAM_MODEL, **cfg)]

show([save_result("imagen completa + SAM", "NMS de mascara", *dedup_masks(segments), sam_model=SAM_MODEL, **cfg)])

SAM vit_h sobre cajas de full: [20/137] 0.9min
SAM vit_h sobre cajas de full: [40/137] 1.7min
SAM vit_h sobre cajas de full: [60/137] 2.4min
SAM vit_h sobre cajas de full: [80/137] 3.1min
SAM vit_h sobre cajas de full: [100/137] 3.9min
SAM vit_h sobre cajas de full: [120/137] 4.7min
SAM vit_h sobre cajas de full: listo en 5.3min


,prueba,variante,florence_size,prompt,num_beams,sam_model,micro_precision,micro_recall,micro_f1,mae_count,total_pred,px_precision,px_recall,dice,union_mask_iou,mean_mask_iou_matched
0,imagen completa + SAM,NMS de mascara,base,grapes,3,vit_h,0.643,0.421,0.509,5.343,1321,0.708,0.945,0.809,0.68,0.83


### SAM `vit_h`: sin cajas de grupo + SAM con NMS de máscara

In [21]:
SAM_MODEL = "vit_h"
cfg = {"phrase": "grapes", "size": "base", "num_beams": 3}
full = detect_full(**cfg)
keep = {name: drop_group_boxes(boxes) for name, boxes in full.items()}

segments_kept = {
    name: {k: v[np.isin(seg["src_idx"], keep[name])] for k, v in seg.items()} for name, seg in segment("full", sam_model=SAM_MODEL, **cfg).items()
}
show([save_result("sin cajas de grupo", "SAM, NMS de mascara", *dedup_masks([segments_kept]), sam_model=SAM_MODEL, **cfg)])

,prueba,variante,florence_size,prompt,num_beams,sam_model,micro_precision,micro_recall,micro_f1,mae_count,total_pred,px_precision,px_recall,dice,union_mask_iou,mean_mask_iou_matched
0,sin cajas de grupo,"SAM, NMS de mascara",base,grapes,3,vit_h,0.702,0.421,0.527,6.029,1213,0.865,0.873,0.869,0.769,0.83


## Comparación

Todas las variantes guardadas en `pruebas/`, incluidas las de sesiones anteriores.

In [22]:
results = pd.DataFrame([json.loads(p.read_text()) for p in sorted(RESULTS_DIR.glob("*.json"))])
results = results.sort_values("micro_f1", ascending=False).reset_index(drop=True)
results.to_csv(OUT_DIR / "comparacion.csv", index=False)
show(results)

,prueba,variante,florence_size,prompt,num_beams,sam_model,micro_precision,micro_recall,micro_f1,mae_count,total_pred,px_precision,px_recall,dice,union_mask_iou,mean_mask_iou_matched
0,sin cajas de grupo,"SAM, NMS de mascara",base,grapes,3,vit_h,0.702,0.421,0.527,6.029,1213,0.865,0.873,0.869,0.769,0.830
1,sin cajas de grupo,"SAM, NMS de mascara",base,grapes,1,vit_b,0.696,0.414,0.519,6.117,1201,0.866,0.837,0.851,0.741,0.816
2,sin cajas de grupo,"SAM, NMS de mascara",base,grapes,3,vit_b,0.690,0.415,0.518,6.022,1214,0.867,0.845,0.856,0.748,0.818
3,sin cajas de grupo,cajas de Florence,base,grapes,3,None,0.676,0.420,0.518,5.723,1255,NaN,NaN,NaN,NaN,NaN
4,sin cajas de grupo,cajas de Florence,base,grapes,1,None,0.676,0.417,0.515,5.803,1244,NaN,NaN,NaN,NaN,NaN
5,imagen completa + SAM,NMS de mascara,base,grapes,3,vit_h,0.643,0.421,0.509,5.343,1321,0.708,0.945,0.809,0.680,0.830
6,imagen completa + SAM,NMS de mascara,base,grapes,3,vit_b,0.628,0.412,0.498,5.255,1325,0.734,0.922,0.817,0.691,0.817
7,imagen completa + SAM,NMS de mascara,base,grapes,1,vit_b,0.626,0.410,0.496,5.299,1323,0.715,0.923,0.806,0.675,0.817
8,prompts,imagen completa,base,grapes,3,None,0.587,0.421,0.490,4.584,1447,NaN,NaN,NaN,NaN,NaN
9,prompts,imagen completa,base,grapes,1,None,0.580,0.418,0.486,4.599,1453,NaN,NaN,NaN,NaN,NaN
